In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive')

import os, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_TRANS = BASE_PATH + "processed_v2/cleaned_transactions.parquet"
INPUT_ARTS = BASE_PATH + "processed_v2/articles_processed.parquet"
OUTPUT_DIR = BASE_PATH + "outputs_v2/candidates/"

spark = SparkSession.builder \
    .appName("Retrieval_Sibling") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
transactions = spark.read.parquet(INPUT_TRANS)
articles = spark.read.parquet(INPUT_ARTS)

max_date = transactions.select(F.max("t_dat_date")).collect()[0][0]

test_start = max_date - datetime.timedelta(days=7)
val_start = test_start - datetime.timedelta(days=7)
train_hist_start = val_start - datetime.timedelta(days=42)
test_hist_start = test_start - datetime.timedelta(days=42)

train_hist_df = transactions.filter((F.col("t_dat_date") >= train_hist_start) & (F.col("t_dat_date") < val_start))
test_hist_df = transactions.filter((F.col("t_dat_date") >= test_hist_start) & (F.col("t_dat_date") < test_start))

In [ ]:
def generate_sibling_candidates(history_df, articles_df, top_n=15):
    history_pc = history_df.withColumn("product_code", F.substring("article_id", 1, 6))

    user_pc = history_pc.select("customer_id", "product_code").dropDuplicates()
    siblings = user_pc.join(articles_df.select("product_code", "article_id"), "product_code", "inner")

    window_spec = Window.partitionBy("customer_id").orderBy(F.col("article_id").desc())

    candidates = siblings.select("customer_id", "article_id").dropDuplicates() \
        .withColumn("rn", F.row_number().over(window_spec)) \
        .filter(F.col("rn") <= top_n) \
        .select("customer_id", "article_id") \
        .withColumn("strategy", F.lit("sibling_product"))
    return candidates

def evaluate_recall(candidates_df, target_df, target_start, target_end):
    actuals = target_df.filter((F.col("t_dat_date") >= target_start) & (F.col("t_dat_date") < target_end)) \
        .select("customer_id", "article_id").dropDuplicates()

    total_actuals = actuals.count()
    hits = actuals.join(candidates_df, ["customer_id", "article_id"], "inner").dropDuplicates().count()
    recall = hits / total_actuals if total_actuals > 0 else 0

    print(f"Actuals: {total_actuals} | Hits: {hits} | Recall: {recall:.4f}")

In [ ]:
train_cands = generate_sibling_candidates(train_hist_df, articles, top_n=15)
train_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "train_sibling.parquet")

test_cands = generate_sibling_candidates(test_hist_df, articles, top_n=15)
test_cands.write.mode("overwrite").parquet(OUTPUT_DIR + "test_sibling.parquet")

print("Evaluating TEST set:")
evaluate_recall(test_cands, transactions, test_start, max_date)

Evaluating TEST set:
Actuals: 207996 | Hits: 4354 | Recall: 0.0209
